# published / calculated / missing NEA parameters
- per parameter: how many `pscomppars` planets have a published value, an NEA-`Calculated Value`, or none (stacks to the full planet count)
- **needs network access** (live NEA TAP queries)
- exports to `report/figures/nea_parameter_provenance.pdf`

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, pathlib

ROOT = pathlib.Path.cwd()
while not (ROOT / 'crossmatching').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import re
import numpy as np
import matplotlib.pyplot as plt
import pyvo

from report_plots import common


In [ ]:
nasa = pyvo.dal.TAPService("https://exoplanetarchive.ipac.caltech.edu/TAP")

columns_query = nasa.search("SELECT column_name FROM TAP_SCHEMA.columns WHERE table_name = 'pscomppars'")
psc_columns = columns_query["column_name"].data
pattern = re.compile(".*ref.*")
ref_columns = [c for c in psc_columns if pattern.match(c)]
len(ref_columns)


In [ ]:
from concurrent.futures import ThreadPoolExecutor

def query_caltech(col):
    query = f"SELECT pl_name, {col} FROM pscomppars WHERE {col} LIKE '%Calculated Value%'"
    return nasa.search(query)

# Use ThreadPoolExecutor to run the synchronous searches concurrently
# Adjust max_workers based on the size of ref_columns to avoid hammering the server
with ThreadPoolExecutor() as executor:
    # executor.map preserves the order of your ref_columns
    calc_queries = list(executor.map(query_caltech, ref_columns))

total_count = int(nasa.search("SELECT COUNT(*) FROM pscomppars")["count(*)"][0])
total_count


In [ ]:
sub_names = {
    "pl_eqt": "Equilibrium\nTemperature",
    "pl_insol": "Insolation",
    "pl_dens": "Density",
    "pl_angsep": "Angular\nSeparation",
    "pl_radj": "Radius",
    "pl_bmassj": "Mass",
    "st_lum": "Host\nStar\nLuminosity",
    "pl_tsm": "Transmission\nSpec.\nMetric",
    "pl_esm": "Emission\nSpec. \nMetric"
}

# Stacked: published / calculated / missing (sums to total_count)
calc_counts = {n.replace("_reflink", ""): len(q) for q, n in zip(calc_queries, ref_columns) if len(q) > 0}
col_order = ["pl_radj", "pl_bmassj", "pl_dens", "st_lum", "pl_insol", "pl_eqt", "pl_angsep", "pl_tsm", "pl_esm"]
stack_names = [sub_names[c] for c in col_order]
stack_calcs = [calc_counts[c] for c in col_order]

def query_missing(col):
    return int(nasa.search(f"SELECT COUNT(*) FROM pscomppars WHERE {col} IS NULL")["count(*)"][0])

with ThreadPoolExecutor() as executor:
    missing = list(executor.map(query_missing, col_order))

published = [total_count - c - m for c, m in zip(stack_calcs, missing)]


In [ ]:
fig, ax = plt.subplots(figsize=(10,3.5))

bottoms = np.zeros(len(stack_names))
for values, label, style in [
    (published, "Published", dict(color="#5E97BC")),
    (stack_calcs, "Calculated", dict(color="#C0DCEB")),
    (missing, "Missing", dict(color="white", hatch="//")),
]:
    bars = ax.bar(stack_names, values, bottom=bottoms, edgecolor="black",
                  linewidth=0.9, label=label, **style)
    seg_labels = [f"{v}" if v > 0.03 * total_count else "" for v in values]
    kwargs = dict(bbox=dict(facecolor="white", edgecolor="none", pad=1)) if "hatch" in style else {}
    ax.bar_label(bars, labels=seg_labels, label_type="center", fontsize=9, **kwargs)
    bottoms += np.array(values)

ax.set_axisbelow(True)
ax.grid(axis="y")
ax.set_ylim(0, total_count)
ax.set_xlim(-0.5, 8.5)
ax.set_yticks([1000*i for i in range(round(total_count/1000)+1)] + [total_count])
handles, leg_labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], leg_labels[::-1], loc="lower right")
common.save_figure("nea_parameter_provenance")
